In [1]:
import warnings
warnings.filterwarnings('ignore')

(Cookbook_Preparing_charged_participants)=
# Preparing charged participants

*Combining stored chemistry, charge-center recognition and whole-center selections.*

Prepare charged participants from an explicit small-molecule state before a
geometric analysis. These SMILES examples require the optional RDKit converter.
The feature recognizer is independent of that backend and does not parameterize.

:::{versionadded} 1.0.0
:::


## Loading explicit chemistry

Zwitterionic glycine demonstrates why total molecular charge does not determine
whether local charged centers exist. Its atom formal charges sum to zero.


In [2]:
import molsysmt as msm
import numpy as np
from molsysmt import pyunitwizard as puw

molsys = msm.convert('smiles:[NH3+]CC(=O)[O-]', to_form='molsysmt.Topology')
formal = msm.get(molsys, element='atom', formal_charge=True)
centers = msm.physchem.get_charge_centers(molsys)
charges = puw.get_value(centers['charges'], to_unit='e')
print('Net formal charge in e:', np.sum(formal))
print('Local center charges in e:', charges)

Net formal charge in e: 0
Local center charges in e: [ 1. -1.]


## Preparing complete selections

Use whole participant membership for chemical selections. Distance-reference
atoms are a separate field: the carboxylate reference omits its central carbon,
but that carbon belongs to the participant. Here we select the whole negative
center on the original system, preserving its atom indices.


In [3]:
negative_center = int(np.flatnonzero(charges < 0)[0])
start, stop = centers['atom_offsets'][negative_center:negative_center + 2]
negative_atoms = centers['atom_indices'][start:stop]
selected = msm.physchem.get_charge_centers(molsys, selection=negative_atoms)
print('Participant atom indices:', selected['atom_indices'])
print('Distance-reference atom indices:', selected['geometry_atom_indices'])
print('Source:', selected['charge_source'], selected['rule_version'])

Participant atom indices: [2 3 4]
Distance-reference atom indices: [3 4]
Source: chemical_states.formal_charge formal_charge_centers@1


## Checking a neutral control

A neutral nitro group has positive and negative atom formal charges, which cancel
within the directly connected charged group. It produces no center in this
bounded definition. A geometric detector must retain this scientific distinction.


In [4]:
molsys = msm.convert('smiles:C[N+](=O)[O-]', to_form='molsysmt.Topology')
neutral = msm.physchem.get_charge_centers(molsys)
print('Neutral control centers:', len(neutral['charges']))
print('Empty membership offsets:', neutral['atom_offsets'])

Neutral control centers: 0
Empty membership offsets: [0]


The feature dictionary does not contain observed interactions or periodic
geometry. The ionic detector is a subsequent delivery; do not attach these
features directly to MolSys.interactions. The current motif rules do not provide
complete classification of phosphates, sulfates or aromatic charged systems.

:::{seealso} Related Tools & References
:class: dropdown

- {func}`molsysmt.basic.convert` — Obtain a native chemical graph.
- {func}`molsysmt.basic.get` — Read stored formal charges.
- {ref}`Tutorial_Get_charge_centers` — Recognize and select charged centers.
- {ref}`user-foundations-native-world-classes-molsysmt-molsys` — Understand chemistry and interaction domains.
:::
